# Algorithm Verification: C&CG vs BDCP

Runs both algorithms on the same instances and flags discrepancies.

**One-time setup** — run from the terminal before opening this notebook:
```bash
cd <folder containing setup.py>
pip install -e .
```

In [1]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
import time
import os

from rcflp.instance  import instancemaker
from rcflp.nominal   import solve_nominal
from rcflp.ccg       import solve_CCG
from rcflp.bdcp      import solve_BDCP
from rcflp.warmstart import solve_robust_warmstart



## 1. Experiment settings

In [2]:
Hn    = 2
Rn    = 3
BIG_M = 10000

# --- Overnight grid ---
I_J_values          = [(15, 10)]#, (15, 10), (20, 15), (25, 15)]
value_max_scales    = [0.75]#, 1.0]
congestion_costs    = [100]#, 100]
uncertainty_budgets = [3]

# Both algorithms use the same 1% relative gap tolerance (matching original code)
OBJ_TOL = 0.01
TIME_MASTER = 500

# Per-instance time limits (set in run loop):
# |I|<=10 -> 5 min,  |I|<=20 -> 15 min,  |I|>20 -> 30 min

parameters = list(itertools.product(
    value_max_scales, congestion_costs, uncertainty_budgets, I_J_values))
print(f"{len(parameters)} instances x 2 algorithms = {2*len(parameters)} solves")


1 instances x 2 algorithms = 2 solves


In [3]:
In = 15
Jn = 10
v = 0.75
w = 100

In [4]:
inst  = instancemaker(In, Jn, Rn, v, w)
nom   = solve_nominal(inst)
x_nom = nom['x_jr']
print(f'  Nominal profit = {nom["profit"]:.1f}')

if In <= 10:
    time_limit = 300
elif In <= 20:
    time_limit = 600
else:
    time_limit = 1800
print(f'  Time limit: {time_limit//60} min per algorithm')

# Assumes inst, x_nom, nom_profit, Hn, BIG_M, OBJ_TOL are already defined
gamma      = 3
time_limit = 900   # 15 minutes per run
 
# Parameter grid
configs = [
    {"label": "A", "tau": 60,  "eps_e": 0.03, "alpha": 0.8, "beta": 120},
#    {"label": "B", "tau": 30,  "eps_e": 0.05, "alpha": 0.8, "beta": 200},
#    {"label": "C", "tau": 100, "eps_e": 0.02, "alpha": 0.8, "beta": 200},
#    {"label": "D", "tau": 50,  "eps_e": 0.04, "alpha": 0.8, "beta": 150},
#    {"label": "E", "tau": 500, "eps_e": 0.03, "alpha": 0.8, "beta": 300},  # baseline with higher eps_e
]
 
# Run loop
rows = []
for cfg in configs:
    print(f"\n{'='*60}")
    print(f"Config {cfg['label']}: tau={cfg['tau']}  eps_e={cfg['eps_e']}  "
          f"alpha={cfg['alpha']}  beta={cfg['beta']}")
    print(f"{'='*60}")
 
    result = solve_CCG(
        inst, gamma, Hn, x_nom,
        tol          = OBJ_TOL,
        big_M        = BIG_M,
        time_limit   = time_limit,
        master_mip_gap    = 0.015,
        master_time_limit = cfg["tau"],
        n_scenarios  = 2,
        L_init       = -nom["profit"],
        eps_e        = cfg["eps_e"],
        alpha        = cfg["alpha"],
        beta         = cfg["beta"],
        verbose      = True,
    )
 
    # Count exploit vs explore iterations
    n_explore = sum(1 for r in result["iter_log"] if r["mode"] == "explore")
    n_exploit = sum(1 for r in result["iter_log"] if r["mode"] == "exploit")
 
    rows.append({
        "config":    cfg["label"],
        "tau":       cfg["tau"],
        "eps_e":     cfg["eps_e"],
        "beta":      cfg["beta"],
        "profit":    round(result["profit_LB"], 2),
        "n_iter":    result["n_iter"],
        "n_explore": n_explore,
        "n_exploit": n_exploit,
        "n_blocks":  result["n_blocks"],
        "runtime_s": round(result["runtime"], 1),
        "converged": result["converged"],
        "final_gap": round(result["iter_log"][-1]["gap_pct"], 2) if result["iter_log"] else None,
    })
 
    print(f"\n  → profit={result['profit_LB']:.1f}  iters={result['n_iter']}  "
          f"explore={n_explore}  exploit={n_exploit}  "
          f"t={result['runtime']:.1f}s  converged={result['converged']}")
 
# Summary table
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
df = pd.DataFrame(rows)
print(df.to_string(index=False))

Set parameter Username
Set parameter LicenseID to value 2785999
Academic license - for non-commercial use only - expires 2027-03-02
  Nominal profit = 173716.7
  Time limit: 10 min per algorithm

Config A: tau=60  eps_e=0.03  alpha=0.8  beta=120
  iter   1 [explore] | L_ell= 173716.75 | UB=      0.00 | gap=30909171459928588.00% | inex=30909171459928588.00% | tau=60s eps_mp=0.0150 | sub=0.0s master=0.0s
         [RAW] L_ell=-173716.7481 UB=-0.0000 U_j=-173716.7481 L_j=-173716.7481 L=-173716.7481
  iter   2 [explore] | L_ell= 173716.75 | UB=  18025.56 | gap=863.72% | inex=863.72% | tau=60s eps_mp=0.0150 | sub=0.2s master=1.0s
         [RAW] L_ell=-173716.7481 UB=-18025.5603 U_j=-173716.7481 L_j=-173716.7481 L=-173716.7481
  iter   3 [explore] | L_ell= 116065.12 | UB=  18025.56 | gap=543.89% | inex=539.92% | tau=60s eps_mp=0.0150 | sub=0.3s master=2.4s
         [RAW] L_ell=-116065.1169 UB=-18025.5603 U_j=-115348.4186 L_j=-116065.1169 L=-115348.4186
  iter   4 [explore] | L_ell=  89070.31 

In [5]:
for row in result["iter_log"]:
    print(f"iter {row['iter']:3d} [{row['mode']:7s}] n_blocks={row['n_blocks']}")

iter   1 [explore] n_blocks=0
iter   2 [explore] n_blocks=1
iter   3 [explore] n_blocks=3
iter   4 [explore] n_blocks=5
iter   5 [explore] n_blocks=7
iter   6 [explore] n_blocks=9
iter   7 [explore] n_blocks=11
iter   8 [explore] n_blocks=13
iter   9 [explore] n_blocks=15
iter  10 [exploit] n_blocks=15
iter  11 [exploit] n_blocks=15
iter  12 [exploit] n_blocks=15
iter  13 [exploit] n_blocks=15
iter  14 [exploit] n_blocks=15
iter  15 [explore] n_blocks=17


## 2. Run both algorithms

## 3. Summary table

In [6]:
display_cols = [
    'I','J','v','w','Gamma','nom_profit',
    'CCG_profit','CCG_iters','CCG_blocks','CCG_time','CCG_conv',
    'BDCP_profit','BDCP_iters','BDCP_time','BDCP_conv',
    'abs_diff','rel_diff_pct','mismatch',
]
df = pd.DataFrame([{c: r[c] for c in display_cols} for r in results])

def highlight(row):
    color = 'background-color: #ffcccc' if row['mismatch'] else ''
    return [color] * len(row)

display(df.style.apply(highlight, axis=1).format(precision=1))
print(f'\nMismatches: {df["mismatch"].sum()} / {len(df)}')
print(f'Timed out (not converged): CCG={len(df[~df["CCG_conv"]])}  BDCP={len(df[~df["BDCP_conv"]])}')

NameError: name 'results' is not defined

## 4. Convergence plots

In [ ]:
to_plot = [r for r in results if r['mismatch']] or results[:1]

for r in to_plot:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"|I|={r['I']} |J|={r['J']} v={r['v']} w={r['w']} Gamma={r['Gamma']}")

    for ax, log, name, color in [
        (axes[0], r['_ccg_log'],  'C&CG', 'steelblue'),
        (axes[1], r['_bdcp_log'], 'BDCP', 'darkorange'),
    ]:
        iters = [d['iter'] for d in log]
        ub    = [-d['UB'] for d in log]
        lb    = [-d['LB'] for d in log]
        ax.plot(iters, ub, label='UB (profit)', color=color, lw=2)
        ax.plot(iters, lb, label='LB (profit)', color=color, lw=2, ls='--')
        ax.set_title(name)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Profit')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 5. Time breakdown: subproblem vs master

In [ ]:
timing_rows = []
for r in results:
    for algo, log in [('CCG', r['_ccg_log']), ('BDCP', r['_bdcp_log'])]:
        total_sub    = sum(d['t_sub']    for d in log)
        total_master = sum(d['t_master'] for d in log)
        total        = total_sub + total_master
        timing_rows.append({
            'I': r['I'], 'J': r['J'], 'v': r['v'], 'w': r['w'], 'Gamma': r['Gamma'],
            'algo':           algo,
            'n_iter':         len(log),
            't_sub_total':    round(total_sub,    1),
            't_master_total': round(total_master, 1),
            'pct_sub':        round(100 * total_sub    / (total + 1e-9), 1),
            'pct_master':     round(100 * total_master / (total + 1e-9), 1),
        })

tdf = pd.DataFrame(timing_rows)
display(tdf)
print('\nAverage % time in subproblem by algorithm:')
display(tdf.groupby('algo')[['pct_sub','pct_master']].mean().round(1))

## 6. Performance comparison (iterations and time)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, label in [
    (axes[0], ('CCG_iters', 'BDCP_iters'), 'Iterations'),
    (axes[1], ('CCG_time',  'BDCP_time'),  'Time (s)'),
]:
    ax.scatter(df[metric[0]], df[metric[1]], c=df['Gamma'], cmap='viridis', s=60, alpha=0.8)
    lim = max(df[metric[0]].max(), df[metric[1]].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', lw=1, label='equal')
    ax.set_xlabel(f'C&CG {label}')
    ax.set_ylabel(f'BDCP {label}')
    ax.set_title(f'{label}: C&CG vs BDCP (colour = Gamma)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()